# Lecture 3.3: Hosted Tools: WebSearchTool, FileSearchTool, CodeInterpreterTool

**Course:** OpenAI Agents SDK: Complete Course  
**Section:** 03: Tools: Extending Agent Capabilities  

---

In this notebook we explore hosted tools. This is a fundamentally different tool category from the `@function_tool` functions covered in Lectures 3.1 and 3.2.

Function tools run in your own Python process. Hosted tools run on OpenAI's servers, alongside the model. You do not write any execution logic. You instantiate a dataclass, drop it in the agent's `tools` list, and OpenAI handles everything else.

By the end of this notebook you will have hands-on experience with the three most commonly used hosted tools.

| Tool | What it does |
|---|---|
| `WebSearchTool` | Gives the agent live web access. No separate search API is needed |
| `FileSearchTool` | Lets the agent query an OpenAI vector store, such as your own uploaded documents |
| `CodeInterpreterTool` | Lets the LLM write and execute Python in a sandboxed OpenAI container |

You will also see how hosted and function tools combine inside a single agent. That combination is the most practical pattern for real-world applications.

## Cell 1 — Install the SDK

📌 **Notebook update notice:** this lecture's markdown references `openai-agents==0.17.7` as the pinned version. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version stated above. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one mentioned in the recording.

The cell below installs the `openai-agents` package. The version is pinned to `0.18.3` for reproducibility, so that every code example in this notebook runs exactly as shown, regardless of when you open it.

To use the latest version instead, run: pip install openai-agents

Or substitute any version you prefer by editing the version string below.

If the package is already installed at this version in your current session, pip will confirm it and move on. Nothing will be re-downloaded.

In [1]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.17.7 as stated in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00


## 2. API Key Setup

This notebook uses Google Colab Secrets to load your OpenAI API key. This keeps your key out of the notebook file and prevents it from being accidentally shared.

### How to add your key in Colab

1. Click the key icon in the left sidebar labelled "Secrets".
2. Click "+ Add new secret".
3. Set the name to exactly: `OPENAI_API_KEY`
4. Paste your OpenAI API key as the value.
5. Toggle "Notebook access" to ON.
6. Run the cell below.

---

Running locally? Set the environment variable in your terminal before launching Jupyter.  
```bash
export OPENAI_API_KEY="your-api-key-here"
```
Then comment out the `userdata` lines and leave only the `os.environ` line if the variable is already set.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 3. Model Name

We declare a single `MODEL_NAME` variable here and use it everywhere an agent is created. Changing this one variable updates the model used across the entire notebook. There is no need to hunt through every `Agent()` call.

The value `"gpt-5.4-mini"` is a good default. It is fast, capable, and cost-effective for experimentation.

To try a different model, edit the string below. See the full list of available models at:  
https://platform.openai.com/docs/models

Note for GPT-5 models: the SDK expects `ModelSettings(reasoning=Reasoning(effort=...))` alongside the model name. All agents in this notebook already include this setting.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## 4. Imports

This cell imports everything needed for the entire notebook.

From the `agents` package, the OpenAI Agents SDK itself:

| Import | Purpose |
|---|---|
| `Agent` | Core agent class. Wraps model, instructions, and tools |
| `CodeInterpreterTool` | Hosted tool for sandboxed Python execution on OpenAI servers |
| `FileSearchTool` | Hosted tool for querying an OpenAI vector store |
| `ModelSettings` | Configures model parameters such as reasoning effort and verbosity |
| `Runner` | Executes an agent run. Use `await Runner.run()` in notebooks |
| `WebSearchTool` | Hosted tool for live web search on OpenAI's infrastructure |
| `function_tool` | Decorator for wrapping Python functions as tools, used in the combination demo |

From the `openai` package, the underlying OpenAI Python SDK:

| Import | Purpose |
|---|---|
| `OpenAI` | Synchronous OpenAI client, used for vector store setup in the `FileSearchTool` demo |
| `CodeInterpreter` (from `openai.types.responses.tool_param`) | The config type that `CodeInterpreterTool` wraps. This is an OpenAI SDK type, not from the agents package |
| `UserLocation` (from `openai.types.responses.web_search_tool`) | Optional location hint for `WebSearchTool`. Also an OpenAI SDK type |
| `Reasoning` (from `openai.types.shared`) | Configures reasoning effort for GPT-5 models |

The separation matters. `WebSearchTool`, `FileSearchTool`, and `CodeInterpreterTool` live in the agents package, while their config sub-types, `UserLocation` and `CodeInterpreter`, come from the openai package.

In [4]:
from openai import OpenAI
from openai.types.responses.tool_param import CodeInterpreter
from openai.types.responses.web_search_tool_param import UserLocation
from openai.types.shared import Reasoning
from agents import (
    Agent,
    CodeInterpreterTool,
    FileSearchTool,
    ModelSettings,
    Runner,
    WebSearchTool,
    function_tool,
)

## 5. Hosted Tools vs Function Tools

Before writing any code, it is worth understanding the execution model distinction, because it changes how you think about tools.

### Function tools (Lectures 3.1 and 3.2)

- You write a Python function and decorate it with `@function_tool`
- The SDK generates a JSON schema from your type annotations
- When the LLM calls the tool, the SDK executes your function in your Python process
- Lifecycle hooks, `on_tool_start` and `on_tool_end`, fire for each call
- `tool_use_behavior`, such as `stop_on_first_tool`, applies

### Hosted tools (this lecture)

- No Python function, no decorator, no schema generation
- You instantiate a dataclass and drop it in the `tools` list
- When the LLM calls the tool, OpenAI's servers execute it. Your Python process is not involved
- `on_tool_start` and `on_tool_end` lifecycle hooks do not fire. They only apply to local tools
- `tool_use_behavior` does not apply. Hosted tools always route through the LLM
- Hosted tools only work with OpenAI models, specifically the `OpenAIResponsesModel` backend. They will fail if you switch to a non-OpenAI provider

### The six hosted tools

| Tool class | What it does |
|---|---|
| `WebSearchTool` | Live web search |
| `FileSearchTool` | OpenAI vector store search |
| `CodeInterpreterTool` | Sandboxed Python execution |
| `HostedMCPTool` | Remote MCP server tools, covered in Update U2 |
| `ImageGenerationTool` | Image generation from prompts |
| `ToolSearchTool` | Deferred tool loading on demand |

This lecture demonstrates the first three in depth. The remaining three are catalogued in the reference table at the end of this notebook.

## 6. WebSearchTool: Basic Usage

`WebSearchTool` gives an agent live access to the web. No configuration is required. `WebSearchTool()` with no arguments is a complete, working tool.

Key points:
- The model decides when to call web search. You do not control the trigger
- The search happens on OpenAI's infrastructure. You are not paying for a separate search API key
- Results are returned to the model, which synthesises them into a final answer
- The `name` property returns `"web_search"`

We use `ModelSettings(reasoning=Reasoning(effort="none"), verbosity="low")` throughout this notebook to keep latency low while experimenting.

In [5]:
agent = Agent(
    name="Web Search Agent",
    instructions=(
        "You are a helpful research assistant. "
        "Use web search to answer questions about current "
        "events and recent information."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[WebSearchTool()],
)

result = await Runner.run(
    agent,
    "What are the latest developments in AI agents in 2026?",
)
print(result.final_output)

Big picture: in 2026, AI agents moved from “demo chatbots” toward **tool-using systems that can do longer, multi-step work** in coding, research, search, and enterprise workflows. OpenAI and Google both highlighted this shift in major 2026 releases and reports. ([openai.com](https://openai.com/index/how-agents-are-transforming-work/?utm_source=openai))

Key developments:
- **Coding agents got much stronger and more widely used.** OpenAI reported heavy Codex agent usage inside OpenAI by June 2026, and a scientific computing field report said agents were significantly accelerating software development and maintenance. ([openai.com](https://openai.com/index/how-agents-are-transforming-work/?utm_source=openai))
- **Agents are becoming core enterprise tools.** OpenAI’s B2B signals showed larger gaps at frontier firms for tools like ChatGPT Agent, Apps in ChatGPT, Deep Research, and GPTs, suggesting companies are increasingly using agents for coding, delegation, and research. ([openai.com](h

## 7. WebSearchTool: With Configuration

`WebSearchTool` accepts several optional parameters that shape how the search is performed.

| Parameter | Type | Default | Effect |
|---|---|---|---|
| `search_context_size` | `"low"` / `"medium"` / `"high"` | `"medium"` | Controls how much web context is retrieved and included in the model's context window |
| `user_location` | `UserLocation` or `None` | `None` | Hints the search engine to bias results toward a geographic location |
| `filters` | `WebSearchToolFilters` or `None` | `None` | Applies attribute-based filters. Advanced usage |
| `external_web_access` | `bool` or `None` | `None` | Set to `False` to request cached or indexed-only behaviour. Live fetch is skipped |

About `UserLocation`. Import it from `openai.types.responses.web_search_tool`. The `type` field must be set to `"approximate"`. This is the only supported value. `country`, `city`, and `region` are optional strings.

About `search_context_size`. `"high"` retrieves more web content, which improves answer quality for complex queries but increases token usage. `"low"` is faster and cheaper. `"medium"` is the default.

The cell below combines both parameters to produce location-aware restaurant suggestions.

In [6]:
agent_located = Agent(
    name="Located Search Agent",
    instructions=(
        "You are a helpful local assistant. "
        "Use web search to answer questions relevant to "
        "the user's location."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[
        WebSearchTool(
            search_context_size="high",
            user_location=UserLocation(
                type="approximate",
                country="IN",
                city="Hyderabad",
                region="Telangana",
            ),
        )
    ],
)

result = await Runner.run(
    agent_located,
    "What are some good places to eat near me today?",
)
print(result.final_output)

Here are a few good **open now** options near Hyderabad today, based on current listings:  

- **Exotica** — North Indian, Biryani, Kebabs; Madhapur; high-rated. ([zomato.com](https://www.zomato.com/hyderabad/best-restaurants?open=now&utm_source=openai))  
- **Café Reed by QUORUM** — cafe / continental / burgers; Hitech City; high-rated. ([zomato.com](https://www.zomato.com/hyderabad/best-restaurants?open=now&utm_source=openai))  
- **TG’s - The Oriental Grill** — Asian / sushi / seafood; Gachibowli; high-rated. ([zomato.com](https://www.zomato.com/hyderabad/best-restaurants?open=now&utm_source=openai))  
- **Roastery Coffee House** — coffee/cafe; Gandipet or Banjara Hills depending on listing; good for a relaxed meal. ([zomato.com](https://www.zomato.com/hyderabad/best-restaurants?open=now&utm_source=openai))  
- **Naatu Kitchen And Bar** — biryani / South Indian / North Indian; Jubilee Hills; budget-friendly. ([zomato.com](https://www.zomato.com/hyderabad/restaurants?cft=2&open=now&u

## 8. FileSearchTool: Vector Store Setup

`FileSearchTool` requires a vector store, an OpenAI-managed index of your documents. The vector store is created using the OpenAI client, not the agents package, and the resulting ID is passed to `FileSearchTool`.

Why set up a vector store this way?
- Files are uploaded to OpenAI's servers and indexed there. No local file handling is required at runtime
- `create_and_poll()` waits until indexing is fully complete before returning, so it is safe to proceed to the agent immediately after this cell
- The `vector_store.id` returned here is what you pass to `FileSearchTool(vector_store_ids=[...])`

What this cell does, step by step:
1. Creates a text document in memory containing facts about the OpenAI Agents SDK
2. Uploads the document to OpenAI Files with `purpose="assistants"`
3. Creates a new vector store named `"agents-sdk-demo"`
4. Adds the uploaded file to the vector store and polls until indexing is complete

Note: this requires an OpenAI account with access to vector stores. Vector store storage costs are listed at https://platform.openai.com/docs/pricing.

In [7]:
import time
client = OpenAI()

sample_text = (
    "The OpenAI Agents SDK was launched in March 2025. "
    "It is the production-ready successor to OpenAI Swarm. "
    "The SDK's four core primitives are: Agents, Tools, "
    "Handoffs, and Guardrails. "
    "It is built on top of the Responses API and supports "
    "Python natively."
)

file_upload = client.files.create(
    file=("agents_sdk_info.txt", sample_text.encode("utf-8")),
    purpose="assistants",
)

vector_store = client.vector_stores.create(
    name="agents-sdk-demo"
)

client.vector_stores.files.create_and_poll(
    vector_store_id=vector_store.id,
    file_id=file_upload.id,
)

print(f"Vector store ready: {vector_store.id}")
time.sleep(10)  # allow the vector store's search index to become fully queryable

Vector store ready: vs_6a7bd8b611708191b4da9eff7f89d0f5


## 9. FileSearchTool: Run the Agent

Now that the vector store is ready, we pass its ID to `FileSearchTool` and run an agent that can query it.

Key parameters:

| Parameter | Value | Effect |
|---|---|---|
| `vector_store_ids` | `[vector_store.id]` | Required. List of vector store IDs to search |
| `max_num_results` | `3` | Cap on how many results the search returns to the model |
| `include_search_results` | `True` | Raw search results are included in the model's context. Useful for citation-style responses |

Advanced parameters not shown here:
- `ranking_options`, to customise the relevance ranking algorithm. Import `RankingOptions` from `openai.types.responses.file_search_tool_param`
- `filters`, to filter results by file attributes. Import `Filters` from the same module

The agent's instructions tell it to answer from the knowledge base only. This is a deliberate constraint to demonstrate retrieval-augmented generation in its simplest form.

In [8]:
agent_searcher = Agent(
    name="Knowledge Base Agent",
    instructions=(
        "You are a helpful assistant. "
        "Answer questions using only the information in "
        "the knowledge base. "
        "If the answer is not in the knowledge base, say so."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[
        FileSearchTool(
            vector_store_ids=[vector_store.id],
            max_num_results=3,
            include_search_results=True,
        )
    ],
)

result = await Runner.run(
    agent_searcher,
    "When was the OpenAI Agents SDK launched and what "
    "are its core primitives?",
)
print(result.final_output)

The OpenAI Agents SDK was launched in March 2025, and its core primitives are Agents, Tools, Handoffs, and Guardrails. 


## 10. CodeInterpreterTool: Sandboxed Code Execution

`CodeInterpreterTool` lets the LLM write and execute Python code. That code runs in a sandboxed container on OpenAI's servers, completely isolated from your local environment.

How to instantiate it:

```python
CodeInterpreterTool(
    tool_config=CodeInterpreter(type="code_interpreter")
)
```

`CodeInterpreter` is imported from `openai.types.responses.tool_param`. The `type="code_interpreter"` field is required. It tells the API which tool variant to activate.

What happens at runtime:
1. The LLM generates Python code to solve the problem
2. That code runs in OpenAI's sandboxed container, not your machine
3. The output, including stdout, results, and any errors, is returned to the model
4. The model interprets the output and writes the final answer

Best use cases for `CodeInterpreterTool`:
- Data analysis and calculations
- Statistical summaries
- CSV and spreadsheet processing
- Mathematical problem solving where showing the working is useful

In [9]:
agent_coder = Agent(
    name="Data Analysis Agent",
    instructions=(
        "You are a data analysis assistant. "
        "Use the code interpreter to perform calculations "
        "and analysis."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[
        CodeInterpreterTool(
            tool_config=CodeInterpreter(
                type="code_interpreter",
                container={"type": "auto"}
            ),
        )
    ],
)

result = await Runner.run(
    agent_coder,
    (
        "Calculate the compound interest on an investment "
        "of $10,000 at 8% annual rate over 10 years. "
        "Show the year-by-year breakdown."
    ),
)
print(result.final_output)

Compound interest formula: **A = P(1 + r)^n**

- **Principal (P):** $10,000
- **Rate (r):** 8% = 0.08
- **Time (n):** 10 years

**Final amount:** **$21,589.25**  
**Total interest earned:** **$11,589.25**

Year-by-year breakdown:

| Year | Interest Earned | Ending Balance |
|---|---:|---:|
| 1 | $800.00 | $10,800.00 |
| 2 | $864.00 | $11,664.00 |
| 3 | $933.12 | $12,597.12 |
| 4 | $1,007.77 | $13,604.89 |
| 5 | $1,088.39 | $14,693.28 |
| 6 | $1,175.46 | $15,868.74 |
| 7 | $1,269.50 | $17,138.24 |
| 8 | $1,371.06 | $18,509.30 |
| 9 | $1,480.74 | $19,990.05 |
| 10 | $1,599.20 | $21,589.25 |

If you want, I can also show this as a simple formula spreadsheet-style table.


## 11. Combining Hosted Tools with Function Tools

One of the most important things to know about hosted tools is that they are completely interoperable with function tools. You can mix both types in the same `tools` list. The model decides which to call based on the task at hand.

Why this matters for real applications:
- Function tools give you access to internal data, such as your databases, APIs, and business logic
- Hosted tools give you access to external capabilities, such as web search, document retrieval, and code execution
- Combining them inside one agent is the pattern you will use most often in production

In the cell below:
- `get_company_context` is a function tool that simulates an internal CRM lookup
- `WebSearchTool()` provides live external market data
- The agent receives both and decides which to call for each part of the query

No special configuration is needed. Just add both to the `tools` list.

In [10]:
@function_tool
def get_company_context(company_name: str) -> str:
    """Returns internal company context for a given company.

    Args:
        company_name: The name of the company to look up.
    """
    return (
        f"Internal data for {company_name}: "
        f"Founded 2015, 250 employees, Series B startup, "
        f"revenue $12M ARR."
    )


agent_combined = Agent(
    name="Research Agent",
    instructions=(
        "You are a business research assistant. "
        "Use web search for current market information and "
        "the company context tool for internal data."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[
        WebSearchTool(),
        get_company_context,
    ],
)

result = await Runner.run(
    agent_combined,
    "Tell me about Acme Corp's market position compared "
    "to current AI market trends.",
)
print(result.final_output)

Acme Corp looks like a **small but plausible AI vendor**: 2015-founded, ~250 employees, Series B, and $12M ARR. In today’s AI market, that means it’s operating in a **very fast-growing but crowded space**. ([gartner.com](https://www.gartner.com/en/newsroom/press-releases/2026-05-19-gartner-forecasts-worldwide-ai-spending-to-grow-47-percent-in-2026?utm_source=openai))

**Market context:**  
- AI spend is still expanding sharply; Gartner forecasts **$2.59T in worldwide AI spending in 2026**, up **47% YoY**. ([gartner.com](https://www.gartner.com/en/newsroom/press-releases/2026-05-19-gartner-forecasts-worldwide-ai-spending-to-grow-47-percent-in-2026?utm_source=openai))  
- The hottest areas are **AI infrastructure, AI software, and agentic workflows**. Gartner says enterprises are increasingly adopting **AI agents** and embedded GenAI, but mostly through **tactical, incremental use cases** rather than broad transformation. ([gartner.com](https://www.gartner.com/en/newsroom/press-releases/

## 12. Hosted Tools: Reference Table

The SDK currently includes six hosted tools. This table summarises all of them for reference.

| Tool | Class | What it does | Key parameters |
|---|---|---|---|
| Web Search | `WebSearchTool()` | Searches the web live | `search_context_size`, `user_location`, `filters`, `external_web_access` |
| File Search | `FileSearchTool(vector_store_ids=[...])` | Searches OpenAI vector stores | `vector_store_ids` (required), `max_num_results`, `include_search_results`, `ranking_options`, `filters` |
| Code Interpreter | `CodeInterpreterTool(tool_config=CodeInterpreter(type="code_interpreter"))` | Executes Python in a sandboxed OpenAI container | `tool_config` (required) |
| Hosted MCP | `HostedMCPTool(tool_config=...)` | Exposes remote MCP server tools to the model | `tool_config`, of type Mcp. Covered in Update U2 |
| Image Generation | `ImageGenerationTool(tool_config=...)` | Generates images from text prompts | `tool_config`, of type ImageGeneration |
| Tool Search | `ToolSearchTool()` | Lets the model load deferred tools on demand | `description`, `execution`, `parameters` |

`HostedMCPTool`, `ImageGenerationTool`, and `ToolSearchTool` are covered in Update Section U2 and advanced content respectively.

## 13. Key Constraints: Important Reminders

Before you use hosted tools in your own projects, keep these constraints in mind.

### Hosted tools only work with OpenAI models
All six hosted tools require the `OpenAIResponsesModel` backend. That means you must use an OpenAI model string, such as `"gpt-5.4-mini"` or `"gpt-5.5"`. If you switch to a non-OpenAI provider, such as Anthropic, Mistral, or LiteLLM, hosted tools will fail. This is by design. The tools run on OpenAI's servers, so a non-OpenAI model has no way to invoke them.

### `tool_use_behavior` does not apply to hosted tools
Settings such as `stop_on_first_tool` or a custom `ToolsToFinalOutputFunction`, covered in Lecture 3.6, affect only function tools. Hosted tools always route back through the LLM. The model always sees the result before producing an output.

### Lifecycle hooks do not fire for hosted tools
`on_tool_start` and `on_tool_end`, covered in Lecture 6.4, apply only to local or function tools. Because hosted tools execute on OpenAI's servers, the SDK has no visibility into their execution. It only sees the result when it comes back.

### Three tools are deferred to later content
- `HostedMCPTool`, covered in Update Section U2, which focuses on the Model Context Protocol
- `ImageGenerationTool`, covered in advanced content
- `ToolSearchTool`, covered in advanced content